#Initialization and Loading

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *
from pyspark.sql.window import Window
import pyspark.sql.functions as F
import pandas as pd
import numpy as np
import datetime

import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger("crm_sales_details_pipeline")

logger.info("Starting Silver Layer Transformation")

#Reading from Bronze table

In [0]:
df = spark.table("workspace.bronze.erp_cust_az12")

logger.info(f"Number of rows (raw): {df.count()}")
logger.info(f"Columns: {df.columns}")
df.display()

#Data quality check

In [0]:
# Nulls check
df_nulls = df.select([count(when(isnull(c), c)).alias(c) for c in df.columns])
logger.info(f"Nulls check: ")
df_nulls.show()

#Duplicate check
dup_count = df.count() - df.dropDuplicates().count()
logger.info(f"Duplicate check: {dup_count}")

            

#Data Transformation

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name))
        )
logger.info(f"String columns trimmed")


##CUSTOMER ID Cleanup



In [0]:
df = df.withColumn(
    "cid",
    F.when(col("cid").startswith("NAS"),
           F.substring(col("cid"), 4, F.length(col("cid"))))
     .otherwise(col("cid"))
)
df.display()

##Birthdate Validation

In [0]:

df = df.withColumn(
    "bdate",
    F.when(col("bdate") > F.current_date(), None)
     .otherwise(col("bdate"))
)

##Gender Normalization

In [0]:
df = df.withColumn(
    "gen",
    F.when(F.upper(col("gen")).isin("F", "FEMALE"), "Female")
     .when(F.upper(col("gen")).isin("M", "MALE"), "Male")
     .otherwise("n/a")
)

##Renaming columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

logger.info(f"Columns renamed")

##Data quality check

In [0]:

df.limit(10).display()

#Writing in Silver Table

In [0]:
logger.info("Writing data to Silver table")

(
    df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.erp_customers")
)

logger.info("Write completed successfully")

## Data validation

In [0]:
%sql
SELECT * From workspace.silver.erp_customers